In [0]:
from pyspark.sql import functions as F

In [0]:
CATALOG = "clinical_trial_supply_chain"

GOLD = f"{CATALOG}.gold"

SOURCE = f"{GOLD}.gold_stockout_risk"

TARGET = (
    f"{GOLD}.gold_replenishment_recommendation"
)

In [0]:
risk_df = spark.table(SOURCE)

In [0]:
recommendation_df = (
    risk_df
    .withColumn(
        "recommended_replenishment_qty",
        F.when(
            F.col("stockout_risk_flag")
            == "HIGH_RISK",

            (
                (
                    F.col("reorder_threshold")
                    *
                    F.col("avg_daily_consumption")
                )
                -
                F.col("available_stock")
            ).cast("int")
        )
        .otherwise(F.lit(0))
    )
    .withColumn(
        "recommendation_status",
        F.when(
            F.col("stockout_risk_flag")
            == "HIGH_RISK",
            "REPLENISH"
        )
        .otherwise(
            "NO_ACTION"
        )
    )
)

In [0]:
(
    recommendation_df.select(
        "site_id",
        "drug_type",
        "available_stock",
        "reorder_threshold",
        "recommended_replenishment_qty",
        "recommendation_status"
    )
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TARGET)
)